# 02 — Missing Analysis

**Amaç:** Missing oranlarını, target ilişkisini ve missing feature adaylarını değerlendirmek.

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT / "src"))
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from IPython.display import display
from kaggle_s6_e7.config import (FIGURES_DIR, TABLES_DIR, TARGET_COL, ID_COL,
    PLOT_SAMPLE_SIZE, RANDOM_STATE, ensure_report_dirs)
from kaggle_s6_e7.data import load_competition_data, infer_feature_columns, validate_schema
ensure_report_dirs()
train, test = load_competition_data()
validate_schema(train, test)
cat_cols, num_cols = infer_feature_columns(train)
plot_df = train.sample(min(PLOT_SAMPLE_SIZE, len(train)), random_state=RANDOM_STATE)
sns.set_theme(style="whitegrid")

## Train/test missing tablosu

In [ ]:
from kaggle_s6_e7.eda import missing_summary, missingness_target_table
missing = missing_summary(train.drop(columns=TARGET_COL), test)
display(missing[missing.train_missing_count.gt(0) | missing.test_missing_count.gt(0)])
missing.to_csv(TABLES_DIR / "02_missing_summary.csv")
missing_cols = missing.index[missing.train_missing_count.gt(0)].tolist()

## Missingness ve target

In [ ]:
tables=[]
for col in missing_cols:
    table=missingness_target_table(train,col,TARGET_COL)
    table.insert(0,"feature",col); tables.append(table.reset_index())
    print(f"\n{col}"); display(table)
missing_target=pd.concat(tables,ignore_index=True)
missing_target.to_csv(TABLES_DIR / "02_missingness_target.csv",index=False)

## Missing count adayı

In [ ]:
from kaggle_s6_e7.features import add_missing_features
feature_cols=[c for c in train.columns if c not in [ID_COL,TARGET_COL]]
train_missing=add_missing_features(train,feature_cols)
missing_count_target=pd.crosstab(train_missing.missing_count,train_missing[TARGET_COL],normalize="index")
display(missing_count_target)
missing_count_target.to_csv(TABLES_DIR / "02_missing_count_target.csv")

## İncelenecek kararlar
- Numeric median + categorical `missing` güvenli V1 mi?
- Hangi kolonların missing flag'i target dağılımını anlamlı değiştiriyor?
- `missing_count` tutulmalı mı?

Sonuçları `eda_findings.md` içine **Gözlem/Aday/Kabul/Red** etiketiyle aktar.